In [1]:
import numpy as np
import pandas as pd
import re
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

from sklearn.dummy import DummyClassifier
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score
import contractions
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [2]:
df = pd.read_parquet('../data/processed/abcd_first_msgs.parquet_v2')

In [3]:
df

,split,id,flow,subflow,first_msgs,refund,refund_label,n_words
0,train,3592,product_defect,return_size,"hi! i need to return an item, can you help me ...",0,1,15
1,train,9489,product_defect,refund_status,just wanted to check on the status of a refund...,0,0,11
2,train,3695,storewide_query,timing_4,hey ho! i've got a promo code and i want to kn...,0,0,27
3,train,5798,shipping_issue,manage,"hello, i would like to change my shipping deat...",0,1,18
4,train,3647,storewide_query,pricing_3,hello. i have a really cool party coming up. a...,0,0,34
...,...,...,...,...,...,...,...,...
10037,test,1622,order_issue,manage_upgrade,hi i want to upgrade an order i have with you ...,0,0,14
10038,test,3293,manage_account,status_credit_missing,"hi, i think i am missing a $40 credit from a s...",0,0,13
10039,test,8492,subscription_inquiry,manage_extension,i would like an extension on my subscription.,0,0,8
10040,test,189,storewide_query,pricing_1,hi! how much do you charge to gift wrap items?,0,0,10


In [4]:
stop_words = set(ENGLISH_STOP_WORDS) - {"no", "not", "nor", "never",'back', 'get', 'give', 'go', 'keep', 'let', 'make', 'put', 'set', 'take', 'turn'}

lemmatizer = WordNetLemmatizer()

GREETINGS_PATTERN = re.compile(
    r"^(?:(?:hi+|hello+|helo+|hey+|hiya|ho|good\s+(morning|afternoon|evening)|there)\b[\s,.!]*)+",
    re.IGNORECASE
)

def clean_message(text):
    if pd.isna(text):
        return ""

    text = str(text).lower().strip()

    text = text.replace("’", "'")

    text = contractions.fix(text)

    text = re.sub(r"\bho!", " ", text)

    text = GREETINGS_PATTERN.sub("", text).strip()

    text = re.sub(r"[^\w\s]|_", " ", text)

    word_tokens = word_tokenize(text, preserve_line=True)

    words = []
    for tokens in word_tokens:
        if tokens not in stop_words and len(tokens) > 1:
            lemma = lemmatizer.lemmatize(tokens, pos="v")
            lemma = lemmatizer.lemmatize(lemma, pos="n")
            words.append(lemma)
    return " ".join(words)

df["clean_text"] = df["first_msgs"].apply(clean_message)

display(df[["first_msgs", "clean_text"]].head(10))

,first_msgs,clean_text
0,"hi! i need to return an item, can you help me ...",need return item help wrong size
1,just wanted to check on the status of a refund...,just want check status refund refund
2,hey ho! i've got a promo code and i want to kn...,get promo code want know expire like use buy h...
3,"hello, i would like to change my shipping deat...",like change ship deatails change recently
4,hello. i have a really cool party coming up. a...,really cool party come need new clothe soon po...
5,i want an extension for my shopping service. i...,want extension shop service need time get money
6,"hello, i am trying to check the status of my o...",try check status order forget username not log...
7,hello i am trying to buy a pair of boots but t...,try buy pair boot not get add cart
8,i accidently ordered the wrong item and was wo...,accidently order wrong item wonder change item...
9,i would like to confirm my address is correct ...,like confirm address correct order able wrong


In [5]:
df

,split,id,flow,subflow,first_msgs,refund,refund_label,n_words,clean_text
0,train,3592,product_defect,return_size,"hi! i need to return an item, can you help me ...",0,1,15,need return item help wrong size
1,train,9489,product_defect,refund_status,just wanted to check on the status of a refund...,0,0,11,just want check status refund refund
2,train,3695,storewide_query,timing_4,hey ho! i've got a promo code and i want to kn...,0,0,27,get promo code want know expire like use buy h...
3,train,5798,shipping_issue,manage,"hello, i would like to change my shipping deat...",0,1,18,like change ship deatails change recently
4,train,3647,storewide_query,pricing_3,hello. i have a really cool party coming up. a...,0,0,34,really cool party come need new clothe soon po...
...,...,...,...,...,...,...,...,...,...
10037,test,1622,order_issue,manage_upgrade,hi i want to upgrade an order i have with you ...,0,0,14,want upgrade order overnight ship
10038,test,3293,manage_account,status_credit_missing,"hi, i think i am missing a $40 credit from a s...",0,0,13,think miss 40 credit subscription refund
10039,test,8492,subscription_inquiry,manage_extension,i would like an extension on my subscription.,0,0,8,like extension subscription
10040,test,189,storewide_query,pricing_1,hi! how much do you charge to gift wrap items?,0,0,10,charge gift wrap item


In [6]:
TARGET = "refund_label"
train = df[df["split"] == "train"]
val = df[df["split"] == "dev"]
test = df[df["split"] == "test"]

X_train = train["clean_text"]
y_train = train[TARGET]

X_val = val["clean_text"]
y_val = val[TARGET]

X_test = test["clean_text"]
y_test = test[TARGET]

In [7]:
print(y_train.value_counts(normalize=True))

dummy = DummyClassifier(strategy="most_frequent")
dummy.fit(X_train, y_train)
y_pred = dummy.predict(X_test)

print("accuracy:", accuracy_score(y_test, y_pred))
print("precision:", precision_score(y_test, y_pred, zero_division=0))
print("recall:", recall_score(y_test, y_pred))

refund_label
0    0.776948
1    0.223052
Name: proportion, dtype: float64
accuracy: 0.7818725099601593
precision: 0.0
recall: 0.0


In [8]:
models = {
    "count + none": make_pipeline(
        CountVectorizer(preprocessor=clean_message, ngram_range=(1, 2)),
        LogisticRegression(max_iter=1000, random_state=42)
    ),
    "count + balanced": make_pipeline(
        CountVectorizer(preprocessor=clean_message, ngram_range=(1, 2)),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    ),
    "tfidf + none": make_pipeline(
        TfidfVectorizer(preprocessor=clean_message),
        LogisticRegression(max_iter=1000, random_state=42)
    ),
    "tfidf + balanced": make_pipeline(
        TfidfVectorizer(preprocessor=clean_message),
        LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)
    )
}

In [9]:
# for name, pipe in models.items():
#     pipe.fit(X_train, y_train)
#     print(f"\n=== {name} ===")
#     print(classification_report(y_val, pipe.predict(X_val)))

In [10]:
results = []

for name, pipe in models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    results.append({
        "model": name,
        "precision": precision_score(y_test, y_pred),
        "recall": recall_score(y_test, y_pred),
        "f1": f1_score(y_test, y_pred),
        'accuracy': accuracy_score(y_test, y_pred)
    })
    results_df = pd.DataFrame(results).sort_values("f1", ascending=False)


In [11]:
results_df


,model,precision,recall,f1,accuracy
1,count + balanced,0.878378,0.890411,0.884354,0.949203
0,count + none,0.933673,0.835616,0.881928,0.951195
3,tfidf + balanced,0.712727,0.894977,0.793522,0.898406
2,tfidf + none,0.839378,0.739726,0.786408,0.912351


In [12]:
best_name = results_df.iloc[0]["model"]
best_model = models[best_name]

In [13]:
best_model.fit(X_train, y_train)

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('countvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function cle...t 0x1188a0860>
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [14]:
df[df["refund_label"] == 1]["subflow"].unique().tolist()

['return_size',
 'manage',
 'status_quantity',
 'refund_update',
 'refund_initiate',
 'return_color',
 'bad_price_competitor',
 'status_mystery_fee',
 'cost',
 'manage_dispute_bill',
 'status_service_added',
 'manage_cancel',
 'return_stain']

In [15]:
prompt = "Hi, i just bought "
refund_prob = best_model.predict_proba([prompt])[0][1]
print(f"Refund request: {refund_prob:.0%}")

Refund request: 45%


In [16]:
X = pd.concat([X_train, X_val], ignore_index=True)
y = pd.concat([y_train, y_val], ignore_index=True)

In [17]:
y.isna().sum()

np.int64(0)

In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report

pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(preprocessor=clean_message)),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])


param_grid = {

    "tfidf__ngram_range": [(1, 1), (1, 2)],
    "tfidf__min_df": [1, 2],
    "tfidf__sublinear_tf": [True, False],


    "clf__C": [0.1, 1.0, 10.0],
    "clf__class_weight": [None, "balanced"],
    "clf__penalty": ["l2"]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="f1",
    n_jobs=-1,
    verbose=1,
    refit=True
)

grid_search.fit(X, y)

Fitting 5 folds for each of 48 candidates, totalling 240 fits


/Users/bangliu/code/MUlbing/ClaimLens/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(
/Users/bangliu/code/MUlbing/ClaimLens/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:1381: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', l1_ratio set to a float between 0 and 1 instead of penalty='elasticnet', and C=np.inf instead of penalty=None.
  warnings.warn(


,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'clf__C': [0.1, 1.0, ...], 'clf__class_weight': [None, 'balanced'], 'clf__penalty': ['l2'], 'tfidf__min_df': [1, 2], ...}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'f1'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",StratifiedKFo... shuffle=True)
,"verbose verbose: int, default=0Controls the verbosity of information printed during fitting, with highervalues yielding more detailed logging.- 0 : no messages are printed;- >=1 : summary of the total number of fits;- >=2 : computation time for each fold and parameter candidate;- >=3 : fold indices and scores;- >=10 : parameter candidate indices and START messages before each fit.",1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.Se

In [25]:
print(f"Best accuracy score:{grid_search.best_score_:.4f}")
print("Best parameters configuration:")
for param, value in grid_search.best_params_.items():
    print(f"  {param}: {value}")

Best accuracy score:0.8982
Best parameters configuration:
  clf__C: 10.0
  clf__class_weight: None
  clf__penalty: l2
  tfidf__min_df: 2
  tfidf__ngram_range: (1, 2)
  tfidf__sublinear_tf: True


In [35]:
y_pre_test = best_model.predict(['i want to have refund for my purchase.'])
y_pre_test

array([1])

In [30]:
best_model

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('countvectorizer', ...), ('logisticregression', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
Name,Type,Value
"classes_ classes_: ndarray of shape (n_classes,)The classes labels. Only exist if the last step of the pipeline is aclassifier.","ndarray[int64](2,)","[0,1]"
,"preprocessor preprocessor: callable, default=NoneOverride the preprocessing (strip_accents and lowercase) stage whilepreserving the tokenizing and n-grams generation steps.Only applies if ``analyzer`` is not callable.",<function cle...t 0x1188a0860>
,"ngram_range ngram_range: tuple (min_n, max_n), default=(1, 1)The lower and upper boundary of the range of n-values for differentword n-grams or char n-grams to be extracted. All values of n suchsuch that min_n <= n <= max_n will be used. For example an``ngram_range`` of ``(1, 1)`` means only unigrams, ``(1, 2)`` meansunigrams and bigrams, and ``(2, 2)`` means only bigrams.Only applies if ``analyzer`` is not callable.","(1, ...)"
,"input input: {'filename', 'file', 'content'}, default='content'- If `'filename'`, the sequence passed as an argument to fit is expected to be a list of filenames that need reading to fetch the raw content to analyze.- If `'file'`, the sequence items must have a 'read' method (file-like object) that is called to fetch the bytes in memory.- If `'content'`, the input is expected to be a sequence of items that can be of type string or byte.",'content'
,"encoding encoding: str, default='utf-8'If bytes or files are given to analyze, this encoding is used todecode.",'utf-8'
,"decode_error decode_error: {'strict', 'ignore', 'replace'}, default='strict'Instruction on what to do if a byte sequence is given to analyze thatcontains characters not of the given `encoding`. By default, it is'strict', meaning that a UnicodeDecodeError will be raised. Othervalues are 'ignore' and 'replace'.",'strict'


In [29]:
y_pred = best_model.predict(X_test)

print(classification_report(y_test, y_pred, digits=4))

              precision    recall  f1-score   support

           0     0.9693    0.9656    0.9675       785
           1     0.8784    0.8904    0.8844       219

    accuracy                         0.9492      1004
   macro avg     0.9238    0.9280    0.9259      1004
weighted avg     0.9495    0.9492    0.9493      1004



In [ ]:
import pandas as pd
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix


linear_svc_pipeline = Pipeline([
    ("tfidf", TfidfVectorizer(
        ngram_range=(1, 2),
        min_df=2,
        sublinear_tf=True
    )),
    ("svc", LinearSVC(
        C=1.0,
        class_weight="balanced",
        random_state=42
    ))
])


cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
y_pred = cross_val_predict(linear_svc_pipeline, X, y, cv=cv, n_jobs=-1)


print("=== Classification Report ===")
print(classification_report(y, y_pred, digits=4))

print("=== Confusion Matrix ===")
print(confusion_matrix(y, y_pred))

=== Classification Report ===
              precision    recall  f1-score   support

           0     0.9731    0.9624    0.9677      7017
           1     0.8742    0.9075    0.8905      2021

    accuracy                         0.9501      9038
   macro avg     0.9236    0.9349    0.9291      9038
weighted avg     0.9509    0.9501    0.9504      9038

=== Confusion Matrix ===
[[6753  264]
 [ 187 1834]]
